# GBDT 如何逐步拟合残差？

**面试回答：**GBDT 在函数空间拟合负梯度；平方损失下负梯度就是残差。每轮浅树只修正上一轮尚未解释的部分，学习率控制修正幅度。

## 真实案例

用距离预测配送分钟数，浅树依次修正远距离订单的系统性低估。

In [1]:
import numpy as np  # 导入 NumPy 手写残差提升。
order=np.array(['D01','D02','D03','D04','D05','D06','V01','V02'])  # 构造订单编号。
x=np.array([1.,2.,3.,5.,6.,8.,4.,7.])  # 记录距离公里数。
y=np.array([18.,21.,25.,36.,41.,53.,30.,47.])  # 记录实际分钟数。
print('订单 | 距离 | 分钟')  # 输出订单表头。
for n,a,b in zip(order,x,y):  # 展示业务样本。
    print(n,a,b)  # 输出一条订单。

订单 | 距离 | 分钟
D01 1.0 18.0
D02 2.0 21.0
D03 3.0 25.0
D04 5.0 36.0
D05 6.0 41.0
D06 8.0 53.0
V01 4.0 30.0
V02 7.0 47.0


## Baseline / 基线

基线始终输出训练订单均值。

In [2]:
train=np.arange(6)  # 选择训练订单。
valid=np.arange(6,8)  # 选择验证订单。
base=np.full(len(valid),y[train].mean())  # 用均值预测验证订单。
base_mse=float(np.mean((base-y[valid])**2))  # 计算基线 MSE。
print('均值基线 MSE=',round(base_mse,2))  # 输出基线。

均值基线 MSE= 110.28


In [3]:
def residual_stump(feature,residual):  # 定义拟合残差的一层回归树。
    candidates=np.unique(feature)[:-1]  # 生成阈值候选。
    scores=[]  # 保存每个切分平方误差。
    for threshold in candidates:  # 枚举阈值。
        left=feature<=threshold  # 构造左右叶。
        prediction=np.where(left,residual[left].mean(),residual[~left].mean())  # 用叶内残差均值预测。
        scores.append(np.sum((residual-prediction)**2))  # 计算残差平方和。
    threshold=candidates[int(np.argmin(scores))]  # 选择最小平方误差阈值。
    left=feature<=threshold  # 重建最优左右叶。
    return threshold,residual[left].mean(),residual[~left].mean()  # 返回树桩叶输出。
def stump_value(feature,tree):  # 定义回归树桩预测。
    threshold,left,right=tree  # 解包树桩参数。
    return np.where(feature<=threshold,left,right)  # 返回叶子值。
pred_train=np.full(len(train),y[train].mean())  # 初始化训练预测为全局均值。
pred_valid=np.full(len(valid),y[train].mean())  # 初始化验证预测为全局均值。
trees=[]  # 保存每轮残差树。
for round_id in range(3):  # 迭代训练三棵浅树。
    residual=y[train]-pred_train  # 计算平方损失负梯度即残差。
    tree=residual_stump(x[train],residual)  # 拟合当前残差。
    trees.append(tree)  # 保存当前树。
    pred_train+=.45*stump_value(x[train],tree)  # 用学习率更新训练预测。
    pred_valid+=.45*stump_value(x[valid],tree)  # 用相同树更新验证预测。
    print('轮次',round_id,'残差均值',round(float(residual.mean()),3),'树',tuple(round(float(v),2) for v in tree))  # 输出逐步残差过程。
mse=float(np.mean((pred_valid-y[valid])**2))  # 计算 GBDT 验证 MSE。

轮次 0 残差均值 -0.0 树 (3.0, -11.0, 11.0)
轮次 1 残差均值 0.0 树 (6.0, -3.14, 15.72)
轮次 2 残差均值 0.0 树 (5.0, -3.44, 6.89)


## 结果解读

第一轮树拟合均值模型的残差，后续树拟合剩余误差；它们不是各自重新预测目标。学习率使单棵树的修正更谨慎。

In [4]:
print('订单 | 真实分钟 | 基线 | GBDT')  # 输出结果表头。
for i,index in enumerate(valid):  # 展示验证预测。
    print(order[index],y[index],round(base[i],1),round(pred_valid[i],1))  # 输出逐订单结果。
print('MSE 基线/GBDT=',round(base_mse,2),round(mse,2))  # 输出指标比较。
print('生产差距：完整 GBDT 需更深树、验证早停、子采样、特征版本与漂移监控。')  # 说明简化边界。

订单 | 真实分钟 | 基线 | GBDT
V01 30.0 32.3 34.3
V02 47.0 32.3 47.5
MSE 基线/GBDT= 110.28 9.43
生产差距：完整 GBDT 需更深树、验证早停、子采样、特征版本与漂移监控。


## 失败案例与修复

若学习率设为 1 且继续大量迭代，浅树会更快追逐训练噪声；修复是 shrinkage 与时间外早停。

In [5]:
fast_train=np.full(len(train),y[train].mean())  # 初始化大步长反例预测。
for tree in trees:  # 复用三棵残差树。
    fast_train+=stump_value(x[train],tree)  # 用学习率一进行激进更新。
print('失败：大步长训练MSE=',round(float(np.mean((fast_train-y[train])**2)),3))  # 输出激进拟合指标。
print('修复：学习率0.45 验证MSE=',round(mse,3))  # 输出带 shrinkage 的回放指标。
print('不能仅用训练MSE判断迭代轮数。')  # 强调验证原则。

失败：大步长训练MSE= 59.953
修复：学习率0.45 验证MSE= 9.431
不能仅用训练MSE判断迭代轮数。


In [6]:
assert len(order)>=5  # 保护样本数量。
assert len(trees)==3  # 保护迭代树数量。
assert mse<base_mse  # 保护提升模型优于均值基线。
assert np.isfinite(pred_valid).all()  # 保护预测数值有限。